# 02 – Text features
Input: `data_clean/reviews_laundry.csv`  
Output: `data_clean/reviews_laundry_features.csv`

Builds attribute flags from a keyword dictionary, then cross-checks with TF-IDF and LDA.

In [ ]:
import re
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

d = pd.read_csv("data_clean/reviews_laundry.csv")
d["text_l"] = d.text.str.lower()
d.shape

## 1. Attribute dictionary
Refine after reading ~100 reviews: `d.sample(100, random_state=1).text`

In [ ]:
ATTR = {
    # product
    "reliability":  r"broke|stopped working|stop working|failed|died|defect|lemon|quit|malfunction|dead",
    "noise":        r"\bloud|noise|noisy|vibrat|rattl|grind|squeal|bang",
    "cleaning":     r"doesn.?t clean|not clean|smell|odor|mold|mildew|residue|stain",
    "design_usability": r"\bdoor|\blid\b|display|\bapp\b|wifi|control panel|button|drawer|cycle|setting",
    "value":        r"price|expensive|overpriced|worth|cheap|money",
    # service
    "repair_service": r"technician|repair|service call|parts|warranty|serviced",
    "support":      r"customer service|customer support|support|\brep\b|on hold|phone call|called lg|called them",
    "delivery_install": r"deliver|install|damaged|scratch|dent|shipping",
    "return_refund": r"refund|return|replace|exchange|money back",
}
for k, pat in ATTR.items():
    d[f"a_{k}"] = d.text_l.str.contains(pat, regex=True).astype(int)
d[[f"a_{k}" for k in ATTR]].mean().round(3).sort_values(ascending=False)

In [ ]:
# Validation: hand-label a sample, then compute precision per attribute.
# 1) export 100 reviews, 2) add a column per attribute with your manual 0/1, 3) compare.
d.sample(100, random_state=1)[["review_id", "text"] + [f"a_{k}" for k in ATTR]].to_csv("data_clean/manual_validation_sample.csv", index=False)

## 2. TF-IDF: terms that separate low from high reviews

In [ ]:
m = d.dropna(subset=["low"])
tf = TfidfVectorizer(stop_words="english", min_df=10, ngram_range=(1, 2), max_features=20000)
X = tf.fit_transform(m.text)
terms = np.array(tf.get_feature_names_out())
low_mean = np.asarray(X[m.low.values == 1].mean(axis=0)).ravel()
high_mean = np.asarray(X[m.low.values == 0].mean(axis=0)).ravel()
diff = low_mean - high_mean
print("LOW:", terms[np.argsort(diff)[-25:][::-1]])
print("HIGH:", terms[np.argsort(diff)[:25]])

## 3. LDA topics (optional cross-check on the dictionary)

In [ ]:
cv = CountVectorizer(stop_words="english", min_df=10, max_df=0.5)
Xc = cv.fit_transform(m.text)
lda = LatentDirichletAllocation(n_components=6, random_state=0).fit(Xc)
w = np.array(cv.get_feature_names_out())
for i, t in enumerate(lda.components_):
    print(i, w[t.argsort()[-12:][::-1]])

In [ ]:
d.drop(columns="text_l").to_csv("data_clean/reviews_laundry_features.csv", index=False)